# NeuroChip Twin v2: CPU reproduction notebook

**Question the twin answers:** measure a new chemical at only a few concentrations on a neural MEA culture,
and forecast its whole dose x developmental-time response (every network feature, every recording day)
with an uncertainty band, before running the remaining wells.

This notebook runs **on CPU, offline**, from the attached dataset *NeuroChip Twin v2 code and bundle*.
It trains nothing and needs no GPU or internet. It:

1. finds the repository snapshot (Kaggle input or a local checkout);
2. checks the Python packages (installs only if one is missing **and** the internet is reachable);
3. verifies the data bundle against its SHA-256 manifest;
4. runs `demo.py` for three held-out chemicals, chosen by a fixed rule (not by hand), and shows the figures;
5. prints the headline result tables straight from `results/*.json`, with their provenance;
6. states the evidence boundaries.

Every result number below is read from files at run time; none is typed into this notebook.

In [ ]:
import os, sys, json, time, re, shutil, zipfile, hashlib, platform, subprocess
from pathlib import Path

T0 = time.time()
os.environ["CUDA_VISIBLE_DEVICES"] = ""            # CPU only, by design (also for the demo subprocesses)
os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")  # the Kaggle input folder is read-only
# Kaggle kernels set KAGGLE_KERNEL_RUN_TYPE; a stray local D:/kaggle/input folder must not count
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or (os.name == "posix" and Path("/kaggle/input").is_dir())
MARKERS = ("demo.py", "data_bundle/manifest.json", "src/neurotwin/__init__.py", "results/trajectory_cv.json")
LOCAL_CHECKOUT = Path("D:/PROJECTS/ai4s-life-sciences/repo")   # authors' machine; ignored elsewhere


def is_repo(p: Path) -> bool:
    return p.is_dir() and all((p / m).is_file() for m in MARKERS)


def walk_dirs(base: Path, max_depth: int = 4):
    """Breadth-first folders under base: Kaggle mounts a dataset at /kaggle/input/<slug>/ (or deeper)."""
    level = [base]
    for _ in range(max_depth + 1):
        nxt = []
        for d in level:
            yield d
            try:
                nxt += sorted(c for c in d.iterdir() if c.is_dir() and not c.name.startswith("."))
            except OSError:
                pass
        level = nxt


def unpack_zipped(d: Path, dest: Path):
    """Fallback when the upload kept sub-folders as <name>.zip archives (kaggle --dir-mode zip, not extracted)."""
    zips = sorted(d.glob("*.zip"))
    if not zips:
        return None
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(d, dest, ignore=shutil.ignore_patterns("*.zip"))
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            has_top = all(n.startswith(z.stem + "/") for n in zf.namelist())
            zf.extractall(dest if has_top else dest / z.stem)
    return next((c for c in walk_dirs(dest, 3) if is_repo(c)), None)


def locate_repo() -> Path:
    cands = [Path(os.environ["NEUROCHIP_REPO"])] if os.environ.get("NEUROCHIP_REPO") else []
    if ON_KAGGLE:
        cands += list(walk_dirs(Path("/kaggle/input")))
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        cands += [p, p / "repo", p / "_kaggle_pkg" / "dataset"]
    cands.append(LOCAL_CHECKOUT)
    for c in cands:
        if is_repo(c):
            return c.resolve()
    if ON_KAGGLE:
        for c in walk_dirs(Path("/kaggle/input")):
            got = unpack_zipped(c, Path("/kaggle/working/neurochip_repo")) if (c / "demo.py").is_file() else None
            if got:
                return got
    raise FileNotFoundError("NeuroChip Twin v2 snapshot not found: attach the Kaggle dataset "
                            "'NeuroChip Twin v2 code and bundle' or set NEUROCHIP_REPO=/path/to/repo")


REPO = locate_repo()
if os.environ.get("NEUROCHIP_OUT"):
    OUT = Path(os.environ["NEUROCHIP_OUT"])
elif ON_KAGGLE:
    OUT = Path("/kaggle/working/neurochip_outputs")
else:
    OUT = REPO / "outputs" / "kaggle_notebook"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO / "src"))
print(f"repository snapshot : {REPO}")
print(f"outputs             : {OUT}")
print(f"running on Kaggle   : {ON_KAGGLE}")

## 1. Environment (CPU only)

Kaggle Python images ship `numpy`, `pandas`, `matplotlib` and `torch`. Nothing is installed unless one of them is
missing **and** the internet is reachable (this kernel is configured with internet off). `torch` is only imported by
the demo subprocesses, so this kernel stays light.

In [ ]:
import importlib.util, importlib.metadata, socket


def requirement_spec(name: str) -> str:
    """Version range for a package, read from the repository's requirements.txt."""
    for line in (REPO / "requirements.txt").read_text(encoding="utf-8").splitlines():
        spec = line.split("#")[0].strip()
        if re.match(rf"{re.escape(name)}\b", spec, flags=re.I):
            return spec
    return name


def online(host: str = "pypi.org", port: int = 443, timeout: float = 3.0) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


NEEDED = ["numpy", "pandas", "matplotlib", "torch"]
missing = [m for m in NEEDED if importlib.util.find_spec(m) is None]
if missing:
    if not online():
        raise RuntimeError(f"missing {missing} and no internet: use a Kaggle Python image, where they are preinstalled")
    specs = [requirement_spec(m) for m in missing]
    print("installing", specs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
ENV = {"python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(),
       "CUDA_VISIBLE_DEVICES": repr(os.environ["CUDA_VISIBLE_DEVICES"]), "packages_installed_now": missing or "none",
       **{m: importlib.metadata.version(m) for m in NEEDED}}
display(pd.Series(ENV, name="value").to_frame())

## 2. Data-bundle integrity

`scripts/verify_bundle.py` recomputes the SHA-256 of every file listed in `data_bundle/manifest.json` (transformed
EPA NFA trajectories and the cross-validation model weights) and fails on any mismatch. When the snapshot was built
by `scripts/make_kaggle_package.py`, every other file is also checked against `kaggle_package_manifest.json`.

In [ ]:
res = subprocess.run([sys.executable, str(REPO / "scripts" / "verify_bundle.py")],
                     capture_output=True, text=True, cwd=OUT)
print((res.stdout + res.stderr).strip())
assert res.returncode == 0, "bundle integrity check failed"

MANIFEST = json.loads((REPO / "data_bundle" / "manifest.json").read_text(encoding="utf-8"))
bundle = pd.DataFrame([{"file": k, "MB": v["bytes"] / 1e6, "source": v.get("source"), "licence": v.get("licence")}
                       for k, v in MANIFEST["files"].items()])
print(f"chemicals: {MANIFEST['n_chemicals']}   wells: {MANIFEST['n_wells']}   "
      f"bundle files: {len(bundle)}   bundle MB: {bundle.MB.sum():.1f}")
display(bundle.groupby(["source", "licence"]).agg(n_files=("file", "size"), MB=("MB", "sum")).round(2))


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


PKG = REPO / "kaggle_package_manifest.json"
if PKG.is_file():
    pkg = json.loads(PKG.read_text(encoding="utf-8"))
    bad = [r for r, m in pkg["files"].items() if not (REPO / r).is_file() or sha256(REPO / r) != m["sha256"]]
    print(f"snapshot: {len(pkg['files']) - len(bad)}/{len(pkg['files'])} files match kaggle_package_manifest.json "
          f"(built {pkg['built_utc']}, git commit {pkg['git_commit']})")
    assert not bad, f"snapshot files differ from the package manifest: {bad[:10]}"
else:
    print("no kaggle_package_manifest.json (running on a plain checkout): bundle check only")

## 3. Held-out forecasts with `demo.py`

For each chemical, `demo.py` loads the ensemble of **that chemical's own cross-validation fold**: models trained
without it, so the chemical is **held-out**. The model is given the wells at *k* measured concentrations spread over
the tested range, forecasts every network feature at every recording day (DIV) for all other concentrations with a
90 % predictive band, and then the real held-out wells are revealed and scored (curve MAE: mean absolute error
against the mean of replicate wells) against two baselines that receive exactly the same measured wells:
log-linear interpolation between the measured concentrations, and an analog k-nearest-neighbour forecast built from
the chemicals of the other folds.

**Which chemicals.** A fixed rule, not a hand pick: from the per-chemical cross-validation results
(`results/trajectory_cv_per_chemical.csv`) at `K_DEMO` measured concentrations, take the chemicals at the 90th, 50th
and 10th percentile of the gain over the best baseline, i.e. a better-than-typical case, a typical case and a
failure case. Set `DEMO_RUNS` by hand to explore other chemicals.

**Abstention flag.** `results/conformal_r7.json` defines the rule: abstain when the epistemic score (mean ensemble
standard deviation of the forecast divided by the mean predictive scale, at the held-out concentrations) exceeds the
90th percentile of the scores of the calibration folds. The per-chemical score and its threshold (cross-fitted on
the chemicals of the other four folds) were computed at build time by `scripts/precompute_site.py` with the same
spread design as `demo.py`; the notebook reads them from `site/data/` after checking that they were built from this
bundle, and says so when they are not available.

Figure colours: blue = forecast and 90 % band, dark = measured (given to the model), red = held out (revealed).

In [ ]:
from neurotwin.models.trajectory import DIVS, FEATURES

TRAJ = json.loads((REPO / "results" / "trajectory_cv.json").read_text(encoding="utf-8"))
print(f"{len(FEATURES)} MEA network features x DIV {DIVS}; ensemble seeds per fold: {TRAJ['seeds']}")
print(f"units: {TRAJ['target_units']}")

K_DEMO = 3                                   # measured concentrations given to the model in the demo runs
QUANTILES = {"better than typical (90th pct)": 0.9, "typical (median)": 0.5, "failure case (10th pct)": 0.1}
DEMO_RUNS = None                             # or e.g. [("Deltamethrin", 3), ("Loperamide", 2)] to override the rule

per_chem = pd.read_csv(REPO / "results" / "trajectory_cv_per_chemical.csv")
best = TRAJ["by_k"][str(K_DEMO)]["curve_mae"]["best_baseline"]
cv_k = (per_chem[per_chem.k == K_DEMO]
        .pivot_table(index=["chemical", "fold", "label"], columns="method", values="curve_mae").reset_index())
cv_k["gain_vs_best"] = cv_k[best] - cv_k["neurotrajectory"]          # > 0: NeuroTrajectory better
cv_k = cv_k.sort_values("gain_vs_best", kind="mergesort").reset_index(drop=True)
agg = cv_k["neurotrajectory"].mean()
print(f"per-chemical CSV consistent with trajectory_cv.json at k={K_DEMO}: mean curve MAE {agg:.4f} vs "
      f"{TRAJ['by_k'][str(K_DEMO)]['curve_mae']['mean']['neurotrajectory']:.4f}; best baseline: {best}; "
      f"chemicals improved: {(cv_k.gain_vs_best > 0).mean():.3f}")
ROLE = {}
if DEMO_RUNS is None:
    DEMO_RUNS = []
    for role, q in QUANTILES.items():
        row = cv_k.iloc[int(round(q * (len(cv_k) - 1)))]
        DEMO_RUNS.append((row.chemical, K_DEMO))
        ROLE[row.chemical] = (role, row.gain_vs_best)
display(cv_k[cv_k.chemical.isin([c for c, _ in DEMO_RUNS])]
        [["chemical", "fold", "label", "neurotrajectory", best, "gain_vs_best"]]
        .rename(columns={"neurotrajectory": "CV curve MAE NeuroTrajectory", best: f"CV curve MAE {best}"})
        .assign(role=lambda d: d.chemical.map(lambda c: ROLE.get(c, ("manual", None))[0]))
        .set_index("chemical").round(3))

runs = []
for chem, k in DEMO_RUNS:
    dest = OUT / "demo" / re.sub(r"[^A-Za-z0-9]+", "_", chem).strip("_")
    t = time.time()
    p = subprocess.run([sys.executable, str(REPO / "demo.py"), "--chemical", chem, "--k", str(k), "--out", str(dest)],
                       capture_output=True, text=True, cwd=OUT)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-4000:])
        raise RuntimeError(f"demo.py failed for {chem}")
    s = json.loads((dest / "summary.json").read_text(encoding="utf-8"))
    runs.append({**s, "k": k, "dir": dest, "wall_s": round(time.time() - t, 1)})
    print(f"{chem}: done in {runs[-1]['wall_s']} s")

SITE = REPO / "site" / "data"


def load_site_scores():
    """Per-chemical epistemic score / threshold / abstain flag precomputed at build time (scripts/precompute_site.py),
    used only if it was built from the same task file and the same R7 rule file as this snapshot."""
    try:
        summ = json.loads((SITE / "summary.json").read_text(encoding="utf-8"))
        index = json.loads((SITE / "index.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as e:
        return None, f"abstention flag not shown: precomputed per-chemical scores not in this snapshot ({type(e).__name__})"
    src = summ.get("sources", {})
    stale = [name for name, path in (("data_bundle/nfa_tasks.npz", REPO / "data_bundle" / "nfa_tasks.npz"),
                                      ("conformal_r7.json", REPO / "results" / "conformal_r7.json"))
             if src.get(name) != sha256(path)]
    if stale:
        return None, f"abstention flag not shown: site/data was built from other versions of {stale}"
    return index, (f"abstention flag: site/data built by {summ.get('generated_by')} at {summ.get('generated_utc')} "
                   f"from this bundle; rule: {index['abstention']['rule']}")


SITE_INDEX, SITE_NOTE = load_site_scores()
SITE_IDS = {c["chem"]: c["id"] for c in SITE_INDEX["chemicals"]} if SITE_INDEX else {}
print(SITE_NOTE)


def site_metrics(chem: str, k: int):
    cid = SITE_IDS.get(chem)
    if cid is None:
        return None
    entry = json.loads((SITE / "chem" / f"{cid}.json").read_text(encoding="utf-8"))["k"].get(str(k))
    return entry["metrics"] if entry else None


for r in runs:
    r["site"] = site_metrics(r["chemical"], r["k"])
same = [abs(r["site"]["curve_mae"]["model"] - r["curve_mae"]["neurotrajectory"]) <= 0.0015 for r in runs if r["site"]]
if same:
    print(f"reproducibility: the build-time precompute and this run give the same NeuroTrajectory curve MAE for "
          f"{sum(same)}/{len(same)} chemicals (same models, same spread design; tolerance 0.0015)")


def flag(s):
    return "n/a" if not s else ("ABSTAIN" if s["abstain"] else "retained")


DEMO_TABLE = pd.DataFrame([{
    "chemical": r["chemical"], "selection": ROLE.get(r["chemical"], ("manual", None))[0],
    "CV fold (never saw it)": r["cv_fold"],
    "EPA DNT reference label": r["epa_dnt_reference_label"], "k measured": r["k"],
    "held-out concentrations": len(r["held_out_concentrations_uM"]),
    "curve MAE NeuroTrajectory": r["curve_mae"]["neurotrajectory"],
    "curve MAE log-linear interp": r["curve_mae"]["loglinear_interp"],
    "curve MAE analog kNN": r["curve_mae"]["analog_knn"],
    "held-out wells in 90% band": r["held_out_wells_inside_90pct_band"],
    "epistemic score": r["site"]["epi"] if r["site"] else None,
    "abstention threshold (other folds)": r["site"]["threshold"] if r["site"] else None,
    "abstention flag": flag(r["site"]),
    "demo wall s": r["wall_s"]} for r in runs]).set_index("chemical")
display(DEMO_TABLE)
print("Demo runs use one deterministic design (measured concentrations spread over the range); the CV numbers "
      "above average several seeded designs, so the two can differ.")
print("Curve MAE in vehicle robust-SD units, lower is better. Single chemicals are illustrations; "
      "the cross-validated evidence is in section 4.")

In [ ]:
for r in runs:
    role = ROLE.get(r["chemical"], ("manual choice", None))[0]
    display(Markdown(f"### {r['chemical']}, k = {r['k']} measured concentrations ({role})\n"
                     f"Held-out: scored with the models of CV fold {r['cv_fold']}, which never saw this chemical.  \n"
                     f"Measured (given to the model), uM: {r['measured_concentrations_uM']}  \n"
                     f"Held out (revealed after the forecast), uM: {r['held_out_concentrations_uM']}  \n"
                     + (f"Abstention flag: **{flag(r['site'])}** (epistemic score {r['site']['epi']:.3f} vs threshold "
                        f"{r['site']['threshold']:.3f} from the other folds)" if r["site"]
                        else "Abstention flag: not available in this snapshot")))
    display(Image(filename=str(r["dir"] / "forecast_curves.png"), width=980))
    display(Image(filename=str(r["dir"] / "trajectory_heatmap.png"), width=680))

## 4. Headline results, read from `results/*.json`

All predictions behind these tables are out-of-fold (chemical-level cross-validation: a chemical is only ever
predicted by models that never saw it). Each table is preceded by its provenance: file, SHA-256, the file's own
description and the code in the snapshot that references it.

In [ ]:
CODE_FILES = sorted([*(REPO / "scripts").glob("*.py"), *(REPO / "src").rglob("*.py")])


def load_result(name: str) -> dict:
    path = REPO / "results" / name
    data = json.loads(path.read_text(encoding="utf-8"))
    refs = [c.relative_to(REPO).as_posix() for c in CODE_FILES
            if c.name != "make_kaggle_package.py" and name in c.read_text(encoding="utf-8", errors="ignore")]
    about = next((data[k] for k in ("description", "title", "method", "scope") if isinstance(data.get(k), str)), "")
    usage = re.search(r"Usage:\s*(.+)", data.get("protocol", ""))
    print(f"[provenance] results/{name}  sha256={sha256(path)[:16]}  bytes={path.stat().st_size}")
    print(f"             about: {about.splitlines()[0] if about else '-'}")
    if usage:
        print(f"             reproduce: {usage.group(1).strip()}")
    print(f"             referenced by: {', '.join(refs) or '-'}")
    return data


ci = lambda c: f"[{c[0]:+.3f}, {c[1]:+.3f}]"
display(Markdown("### R2: few-shot dose x DIV trajectory forecasting of unseen chemicals (curve MAE, lower is better)"))
R2 = load_result("trajectory_cv.json")
print(f"chemicals: {R2['n_chemicals']}; outer folds: {len(R2['outer_folds'])}; seeds: {R2['seeds']}; "
      f"designs per k: {R2['designs_per_k']}")
rows = []
for k, v in sorted(R2["by_k"].items(), key=lambda kv: int(kv[0])):
    c, w = v["curve_mae"], v.get("well_interval90", {})
    best = c["best_baseline"]
    rows.append({"k measured": int(k), "NeuroTrajectory": c["mean"]["neurotrajectory"],
                 "log-linear interp": c["mean"]["loglinear_interp"], "analog kNN": c["mean"]["analog_knn"],
                 "best baseline": best, "diff vs best": c["paired_vs_best"]["mean_diff"],
                 "95% CI (bootstrap)": ci(c["paired_vs_best"]["ci95"]),
                 "rel. change %": c["paired_vs_best"]["rel_change_pct"],
                 "chemicals improved": c["paired_vs_best"]["frac_chem_improved"],
                 "n chemicals": c["paired_vs_best"]["n_chemicals"],
                 "90% interval coverage (wells)": w.get("coverage_mean"), "interval width": w.get("width_mean")})
display(pd.DataFrame(rows).set_index("k measured"))
print("Sensitivity:", R2["development_note"])
display(pd.DataFrame([{"k measured": int(k), "NeuroTrajectory": v["neurotrajectory"], "best baseline": v["best_baseline"],
                       "baseline": v["baseline"], "diff vs best": v["paired_vs_best"]["mean_diff"],
                       "95% CI (bootstrap)": ci(v["paired_vs_best"]["ci95"]),
                       "rel. change %": v["paired_vs_best"]["rel_change_pct"],
                       "chemicals improved": v["paired_vs_best"]["frac_chem_improved"],
                       "n chemicals": v["paired_vs_best"]["n_chemicals"]}
                      for k, v in sorted(R2["by_k_folds_1to4"].items(), key=lambda kv: int(kv[0]))]
                     ).set_index("k measured"))

In [ ]:
display(Markdown("### R7: conformal calibration of the predictive intervals, and abstention"))
R7 = load_result("conformal_r7.json")
rows = []
for a, block in R7["results"].items():
    for k, v in block.items():
        if not k.startswith("k"):
            continue
        rows.append({"nominal": block["nominal"], "k measured": int(k[1:]),
                     "parametric coverage": v["parametric"]["coverage"], "parametric width": v["parametric"]["width"],
                     "conformal coverage": v["conformal"]["coverage"],
                     "conformal 95% CI": f"[{v['conformal']['ci95'][0]:.3f}, {v['conformal']['ci95'][1]:.3f}]",
                     "conformal width": v["conformal"]["width"],
                     **{f"coverage {g}": s["conformal_coverage"] for g, s in v.get("by_regime", {}).items()}})
display(pd.DataFrame(rows).set_index(["nominal", "k measured"]))
print("chemicals per cytotoxicity regime:", R7["chemicals_per_regime"])
ab = R7["abstention"]
print(f"abstention rule: {ab['rule']}")
display(pd.DataFrame([{"k measured": "all", "abstention rate": ab["abstention_rate"],
                       "curve MAE retained": ab["curve_mae_retained"], "curve MAE abstained": ab["curve_mae_abstained"]}]
                     + [{"k measured": k, "abstention rate": v["rate"], "curve MAE retained": v["mae_retained"],
                         "curve MAE abstained": v["mae_abstained"]} for k, v in ab["by_k"].items()]).set_index("k measured"))

In [ ]:
display(Markdown("### R8: directed propagation on the real Brewer 4-compartment MEA (chip layer)"))
R8 = load_result("r8_chiplayer.json")
cv = R8["cv"]
print(f"source: {R8['source']} ({R8['license']}); recordings: {R8['n_recordings']} {R8['conditions']}; "
      f"sorted axons: {R8['n_axons']}")
print(f"graph: {R8['graph_description']}")
print(f"protocol: {cv['protocol']}; model: {cv['model']}; aggregation: {cv['aggregation']}")
display(pd.DataFrame(cv["summary"]).T.rename_axis("method").round(4))
display(pd.DataFrame([{"baseline": b, "metric": m, "model - baseline": v["difference_model_minus_baseline"],
                       "95% CI": ci(v["ci95"]), "p (no improvement)": v["p_bootstrap_no_improvement"],
                       "unit": v["unit"], "resamples": v["n_resamples"]}
                      for b, mets in cv["comparisons"].items() for m, v in mets.items()]).set_index(["baseline", "metric"]))
print("note:", cv["direction_null_delay_note"])

PAR = load_result("chiplayer_params.json")
print(f"cohort: {PAR['source_cohort']}; scope: {PAR['scope']}; units: {PAR['units']}")
display(pd.DataFrame([{"pair": e["pair"], "axons FF": e["n_axons_ff"], "axons FB": e["n_axons_fb"],
                       "directionality d": e["observed_directionality_d"], "FF weight": e["ff_weight"],
                       "conduction median ms": e["conduction"]["median_ms"],
                       "conduction p10-p90 ms": f"{e['conduction']['p10_ms']:.2f}-{e['conduction']['p90_ms']:.2f}"}
                      for e in PAR["fit_pair_edges"]]).set_index("pair").round(3))
for fig in sorted((REPO / "results" / "figures").glob("chip_*.png")):
    display(Markdown(f"`results/figures/{fig.name}`"))
    display(Image(filename=str(fig), width=760))

In [ ]:
display(Markdown("### Reference: reproduction of the published EPA NFA bioactivity baseline"))
EPA = load_result("epa_baseline_repro.json")
print(f"method: {EPA['method']}; fit stage: {EPA['endpoint_fit_stage']}")
print(f"reference: {EPA['reference_revision']} (n={EPA['n_reference']}: {EPA['n_positive']} positive, "
      f"{EPA['n_negative']} negative); upstream commit {EPA['upstream_commit']}")
display(pd.DataFrame([{"model": m, "hits needed": v["threshold_hits"], "TP": v["tp"], "FN": v["fn"], "TN": v["tn"],
                       "FP": v["fp"], "sensitivity %": v["sensitivity_pct"], "specificity %": v["specificity_pct"],
                       "balanced accuracy %": v["balanced_accuracy_pct"],
                       "delta vs published sens/spec/BA (pp)": "/".join(
                           str(v["published_delta_pp"][x]) for x in ("sensitivity_pp", "specificity_pp", "balanced_accuracy_pp"))}
                      for m, v in EPA["models"].items()]).set_index("model"))
print("comparison:", EPA["comparison_explanation"])
print("interpretation:", EPA["interpretation"])

## 5. Evidence boundaries

- **What the data are.** EPA NFA: rat primary cortical cultures in 48-well MEA plates, recorded while the network
  develops. This is **not a validated twin of any commercial organ-on-a-chip**.
- **Held-out is chemical-level.** Each chemical is scored only with the models of its own cross-validation fold,
  which never saw it; errors are measured against real wells at concentrations the model was not given.
- **Chip layer.** Directed propagation between compartments is fitted and cross-validated on the real Brewer
  4-compartment MEA recordings; any compartment scenario built on it is a **simulation**, not a measurement.
- **Not clinical.** No human data and no clinical or regulatory decision; EPA reference labels are shown only as
  context.
- **Uncertainty.** The demo band is the ensemble's parametric 90 % band; calibrated (conformal) coverage and the
  abstention rule are in table R7.

The scope statements stored with the results are printed below.

In [ ]:
SCOPE = {
    "target units (R2)": R2["target_units"],
    "abstention rule (R7)": R7["abstention"]["rule"],
    "chip parameters scope (R8)": PAR["scope"],
    "burst analysis (R8)": R8["burst_interpretation"],
    "recording ids (R8)": R8["fid_semantics"],
    "EPA baseline (reference)": EPA["interpretation"],
}
for k, v in SCOPE.items():
    print(f"- {k}: {v}")

REPORT = {"repo": str(REPO), "on_kaggle": ON_KAGGLE, "gpu_used": False, "env": ENV,
          "demo_runs": [{kk: (str(vv) if isinstance(vv, Path) else vv) for kk, vv in r.items()} for r in runs],
          "wall_s": round(time.time() - T0, 1)}
(OUT / "run_report.json").write_text(json.dumps(REPORT, indent=2, default=str), encoding="utf-8")
print(f"\nTotal wall time on CPU: {REPORT['wall_s']} s  (report: {OUT / 'run_report.json'})")